# Generative AI: exposure and adoption across economic divides

This notebook builds the eight static charts for the project:

Every chart is saved to `figures/` as a PNG; the last cell also saves SVGs for `make_report.py`.
**Run it top to bottom** (Restart Kernel → Run All): later cells depend on variables made earlier.

## Setup
Imports, shared constants and the chart theme used by every chart.

In [ ]:
from pathlib import Path

import altair as alt   # charts
import numpy as np     # correlation, conditional columns
import pandas as pd    # data tables
import pycountry       # converts ISO3 codes ("USA") to names and numeric codes (840) for the map

# Folder with the raw CSVs, relative to this notebook (static-project/)
DATA_DIR = Path("data/raw")
Path("figures").mkdir(exist_ok=True)   # where every chart is saved

# The nine ISCO-08 major occupation groups, in official order 1-9.
# Charts sort by this list so occupations appear in that order rather than alphabetically.
ISCO = ["Managers", "Professionals", "Technicians & associate professionals",
        "Clerical support workers", "Service & sales workers", "Skilled agricultural workers",
        "Craft & related trades", "Plant & machine operators", "Elementary occupations"]
CLERICAL = "Clerical support workers"   # the group the ILO finds most exposed to generative AI

# World Bank income groups, low to high
INCOME = ["Low income", "Lower middle income", "Upper middle income", "High income"]

# Colors
BLUES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]  # light -> dark
BLUE, GRAY = "#2a78d6", "#a3a29c"        # main series color, and gray for averages / reference bars

# Show charts as static images in the notebook (matches the exported files)
alt.renderers.enable("png")


@alt.theme.register("course", enable=True)
def theme():
    """Shared look for every chart: off-white background, left-aligned titles, quiet gray axes."""
    gray = {"labelColor": "#52514e", "titleColor": "#52514e"}
    return {"config": {"background": "#fcfcfb", "view": {"stroke": None},
                       "title": {"anchor": "start", "subtitleColor": "#52514e"},
                       "axis": {**gray, "gridColor": "#e8e7e3", "domainColor": "#e8e7e3"},
                       "legend": gray}}

## Part 1: Workforce exposure (ILOSTAT)

### Load and clean the occupation data
Each ILOSTAT row is employment (in thousands) for one country, survey, sex, occupation group and year.
The file lists the same workers under several classifications, so we keep only the nine ISCO-08 groups and
one survey per country-year to avoid counting anyone twice.

In [ ]:
occ = pd.read_csv(DATA_DIR / "ilostat_emp_occ_1digit.csv", low_memory=False)

# Keep only ISCO-08 groups 1-9 (drops armed forces "_0", unclassified "_X", ISCO-88 and skill levels)
occ = occ[occ["classif1"].str.match(r"^OCU_ISCO08_[1-9]$")].rename(
    columns={"ref_area": "iso3", "time": "year", "obs_value": "emp"})

# Some countries report the same year from two surveys; keep the first one listed for each country-year
occ = occ[occ["source"] == occ.groupby(["iso3", "year"])["source"].transform("first")]

# Readable labels: "OCU_ISCO08_4" -> "Clerical support workers", "SEX_F" -> "Women"
occ["occupation"] = occ["classif1"].str[-1].astype(int).map(lambda i: ISCO[i - 1])
occ["sex"] = occ["sex"].map({"SEX_T": "Total", "SEX_M": "Men", "SEX_F": "Women"})

# Each group's share of employment for that country, year and sex.
# The denominator is the nine groups only (armed forces and unclassified workers are excluded).
occ["share"] = occ["emp"] / occ.groupby(["iso3", "year", "sex"])["emp"].transform("sum")

# Country names and numeric codes (the world map matches countries on the numeric code)
countries = {c.alpha_3: (getattr(c, "common_name", c.name), int(c.numeric)) for c in pycountry.countries}
occ["country"] = occ["iso3"].map(lambda c: countries.get(c, (c, None))[0])
occ["iso_num"] = occ["iso3"].map(lambda c: countries.get(c, (c, None))[1])

# Add World Bank income groups. The inner join also drops regional aggregates like "World".
occ = occ.merge(pd.read_csv(DATA_DIR / "worldbank_countries.csv")[["iso3", "income_group"]],
                on="iso3", how="inner")

# Each country's most recent year. Named occ_latest (not latest) so the OECD section can't overwrite it.
occ_latest = occ[occ["year"] == occ.groupby("iso3")["year"].transform("max")]
total = occ_latest[occ_latest["sex"] == "Total"]   # both sexes combined: used by the map and heatmap

### Chart 1: World map of clerical employment
Share of each country's employment in clerical jobs, the occupation group most exposed to generative AI.
Countries without ISCO-08 data stay gray, with their own legend entry so gray reads as "missing", not "zero".

In [ ]:
m = total[total["occupation"] == CLERICAL]
mapped = m[["iso_num", "share"]].astype({"iso_num": int})   # integer codes to match the map's ids
yr_min, yr_max = m["year"].min(), m["year"].max()           # range of "latest years", for the subtitle

# Country shapes (Natural Earth, 1:110m), loaded from the web when the chart renders
world = alt.topo_feature("https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/world-110m.json", "countries")
NO_DATA = "No occupation data"

# Layer 1: every country in gray, with a one-item legend explaining what gray means
gray = alt.Chart(world).mark_geoshape(stroke="white", strokeWidth=0.5).transform_calculate(
    status=f"'{NO_DATA}'"
).encode(color=alt.Color("status:N", title=None,
                         scale=alt.Scale(domain=[NO_DATA], range=["#d9d8d3"]),
                         legend=alt.Legend(orient="bottom", symbolType="square", symbolSize=200)))

# Layer 2: countries with data, colored by clerical share.
# The threshold scale bins values into 5 classes (<4%, 4-8%, 8-12%, 12-16%, 16%+) so differences are visible.
shares = (
    alt.Chart(world).mark_geoshape(stroke="white", strokeWidth=0.5)
    .transform_lookup(lookup="id", from_=alt.LookupData(mapped, "iso_num", ["share"]))  # join data to shapes
    .transform_filter("isValid(datum.share)")                                          # keep only matched
    .encode(color=alt.Color(
        "share:Q", title="Share of employment in clerical jobs",
        scale=alt.Scale(type="threshold", domain=[0.04, 0.08, 0.12, 0.16], range=BLUES[1:6]),
        legend=alt.Legend(format=".0%", orient="bottom", direction="horizontal",
                          titleLimit=300, gradientLength=300)))
)

map_chart = (
    (gray + shares)
    .transform_filter("datum.id != 10")   # drop Antarctica (id 10): no data, and it takes a lot of space
    .project("equalEarth")                # equal-area projection, so country sizes aren't distorted
    .resolve_scale(color="independent")   # two separate color legends: "no data" and the share scale
    .properties(
        width=760, height=400,
        title=alt.Title(
            "Clerical jobs, the work most exposed to generative AI, are concentrated in high-income countries",
            subtitle=[f"Share of employment in clerical support occupations (ISCO-08 group 4), latest year available ({yr_min}–{yr_max}).",
                      "Exposed = a large share of the job's tasks could potentially be done by generative AI (ILO, 2025); "
                      "exposure is not the same as job loss.",
                      "Gray = no occupation data reported to the ILO, including large economies such as China and Canada."]))
)
map_chart.save("static-viz/figures/clerical_map.png", scale_factor=2)   # scale_factor=2 doubles the resolution
map_chart

### Chart 2: Occupation mix by income group
How employment splits across the nine occupation groups in each World Bank income group.
Each column sums to 100%. The clerical row is outlined to tie it to the rest of the project.

In [ ]:
# Add up employment across countries before taking shares, so each column is weighted by workforce size
# ("what does the typical worker's job look like in this income group?")
inc = total.groupby(["income_group", "occupation"], as_index=False)["emp"].sum()
inc["share"] = inc["emp"] / inc.groupby("income_group")["emp"].transform("sum")

# Column labels with the number of countries behind each, e.g. "Low income (n=18)"
n = total.groupby("income_group")["iso3"].nunique()
inc["income_label"] = inc["income_group"] + " (n=" + inc["income_group"].map(n).astype(str) + ")"
label_order = [f"{g} (n={n[g]})" for g in INCOME if g in n]   # low -> high

# Bold axis label for the clerical row only
bold_clerical = {"condition": {"test": f"datum.value == '{CLERICAL}'", "value": "bold"}, "value": "normal"}

base = alt.Chart(inc).encode(
    x=alt.X("income_label:N", sort=label_order, title=None,
            axis=alt.Axis(labelAngle=0, orient="top", ticks=False, domain=False)),
    y=alt.Y("occupation:N", sort=ISCO, title="Occupation group (ISCO-08)",
            axis=alt.Axis(ticks=False, domain=False, labelFontWeight=bold_clerical)))

# Colored cells. The scale is capped at 35% so the large agriculture values don't wash out everything else.
cells_ = base.mark_rect(stroke="white", strokeWidth=2).encode(
    color=alt.Color("share:Q", legend=None,
                    scale=alt.Scale(range=BLUES, domain=[0, 0.35], clamp=True)))

# Percentages printed in each cell; white text on dark cells, black on light ones
values = base.mark_text(fontSize=12).encode(
    text=alt.Text("share:Q", format=".0%"),
    color=alt.condition("datum.share > 0.22", alt.value("white"), alt.value("#0b0b0b")))

# Black outline around the clerical row (x and x2 stretch it across the full chart width)
outline = base.transform_filter(f"datum.occupation == '{CLERICAL}'").mark_rect(
    fill=None, stroke="#0b0b0b", strokeWidth=2.5).encode(x=alt.value(0), x2=alt.value(560))

heatmap = (cells_ + values + outline).properties(
    width=560, height=320,
    title=alt.Title(
        "As income rises, work shifts from farms to offices",
        subtitle=["Share of employment by occupation; each column sums to 100%. Outlined = clerical work, the group most exposed to GenAI.",
                  "Countries with ISCO-08 data, latest year available, weighted by workforce size."]))
heatmap.save("static-viz/figures/occupation_heatmap.png", scale_factor=2)
heatmap

### Chart 3: Women vs. men in clerical jobs
For each country, the share of employed women and of employed men who work in clerical jobs.
Shows the 20 countries with the largest gap, among countries with at least 1 million workers
(small countries can have noisy figures). Sorted from largest to smallest gap.

In [ ]:
TOP_N = 20                          # how many countries to show
WOMEN, MEN = "#7b4fd0", "#e09b00"   # purple / amber: distinct from the blue-orange used elsewhere

# Countries with at least 1 million workers (emp is in thousands)
workers = total.groupby("iso3")["emp"].sum()
big = workers[workers >= 1000].index

# One row per country with "Women" and "Men" columns, then the gap between them
cler = (occ_latest[(occ_latest["occupation"] == CLERICAL) & occ_latest["sex"].isin(["Women", "Men"])
                   & occ_latest["iso3"].isin(big)]
        .pivot_table(index="country", columns="sex", values="share").reset_index().dropna())
cler = cler.assign(gap=cler["Women"] - cler["Men"]).nlargest(TOP_N, "gap")

# Shared axes for all layers
y = alt.Y("country:N", sort=alt.SortField("gap", order="descending"), title=None,
          axis=alt.Axis(ticks=False, domain=False))
x_scale = alt.Scale(domain=[0, 0.32], nice=False)

# Gray line connecting men's and women's values
rule = alt.Chart(cler).mark_rule(color="#c9c8c2", strokeWidth=2).encode(
    y=y, x=alt.X("Men:Q", scale=x_scale), x2="Women:Q")

# The two dots. transform_fold turns the Women/Men columns into rows so they can share one color legend.
dots = alt.Chart(cler).transform_fold(["Women", "Men"], as_=["sex", "share"]).mark_circle(size=90, opacity=1).encode(
    y=y,
    x=alt.X("share:Q", scale=x_scale,
            title="Share of employed women / men working in clerical jobs",
            axis=alt.Axis(format=".0%", values=[0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30])),
    color=alt.Color("sex:N", title=None, scale=alt.Scale(domain=["Women", "Men"], range=[WOMEN, MEN]),
                    legend=alt.Legend(orient="top")))

# Gap labels ("+17 pts") only for the 3 largest and 3 smallest gaps; the rest can be read from the axis
ends = pd.concat([cler.nlargest(3, "gap"), cler.nsmallest(3, "gap")])
gap_labels = alt.Chart(ends).transform_calculate(
    gap_label="'+' + format(datum.gap * 100, '.0f') + ' pts'"
).mark_text(align="left", dx=8, fontSize=10, color="#52514e").encode(
    y=y, x=alt.X("Women:Q", scale=x_scale), text="gap_label:N")

dumbbell = (rule + dots + gap_labels).properties(
    width=500, height=alt.Step(18),   # Step(18) = 18 pixels per country row
    title=alt.Title(
        "Women are far more likely than men to work in clerical jobs",
        subtitle=[f"The {TOP_N} economies with the largest gap, among those with 1M+ workers, sorted from largest to smallest gap.",
                  "Latest year available. Clerical work is the occupation group most exposed to generative AI (ILO, 2025)."]))
dumbbell.save("static-viz/figures/sex_clerical_jobs.png", scale_factor=2)
dumbbell

### Chart 4: Clerical share over time, by income group
Average clerical share of employment in each World Bank income group, 2012–2024.
Each country counts equally. To stop the averages jumping when countries enter or leave the data, only
countries with at least 10 of the 13 years are used, and their missing years are filled in by interpolation.

In [ ]:
cl = occ[(occ["occupation"] == CLERICAL) & (occ["sex"] == "Total") & occ["year"].between(2012, 2024)]

# Keep countries with data in at least 10 of the 13 years
n_years = cl.groupby("iso3")["year"].nunique()
cl = cl[cl["iso3"].isin(n_years[n_years >= 10].index)]

# Table of years x countries; fill the few missing years by straight-line interpolation
# (at the start or end of a series, the nearest available value is used)
panel = (cl.pivot_table(index="year", columns="iso3", values="share")
         .reindex(range(2012, 2025)).interpolate(limit_direction="both"))

# Back to long format, add each country's income group, and average within each group and year
group_of = cl.drop_duplicates("iso3").set_index("iso3")["income_group"]
ts = (panel.reset_index().melt(id_vars="year", var_name="iso3", value_name="share")
      .assign(income_group=lambda d: d["iso3"].map(group_of))
      .groupby(["income_group", "year"], as_index=False)["share"].mean())

# Line labels with the number of countries, e.g. "High income (n=40)", ordered high to low
n = group_of.value_counts()
ts["group"] = ts["income_group"] + " (n=" + ts["income_group"].map(n).astype(str) + ")"
order = [f"{g} (n={n[g]})" for g in reversed(INCOME) if g in n]
shades = ["#0d366b", "#256abf", "#3987e5", "#6da7ec"]   # darker = higher income

x = alt.X("year:Q", title=None, scale=alt.Scale(domain=[2012, 2024], nice=False),
          axis=alt.Axis(format="d", tickMinStep=1))      # format="d" shows 2012, not 2,012
y = alt.Y("share:Q", title="Average share of employment", scale=alt.Scale(domainMin=0),
          axis=alt.Axis(format=".0%"))
color = alt.Color("group:N", sort=order, legend=None, scale=alt.Scale(domain=order, range=shades[:len(order)]))

lines = alt.Chart(ts).mark_line(point=True, strokeWidth=2.5).encode(x=x, y=y, color=color)

# Label each line at its right end instead of using a legend
end_labels = alt.Chart(ts[ts["year"] == 2024]).mark_text(
    align="left", dx=8, fontSize=11, fontWeight="bold").encode(x=x, y=y, text="group:N", color=color)

line_chart = (lines + end_labels).properties(
    width=560, height=300,
    title=alt.Title(
        "Clerical work is a far larger share of jobs in high-income countries",
        subtitle=["Average share of employment in clerical support occupations, by World Bank income group, 2012–2024.",
                  "Each country counts equally; countries with at least 10 years of data, short gaps interpolated."]))
line_chart.save("static-viz/figures/clerical_lines.png", scale_factor=2)
line_chart

## Part 2: Business AI adoption (OECD)

### Load and clean the OECD data
Measure `G14_B` is the share of enterprises using AI, by country, industry, firm size and year.
Only data from 2023 onward is kept: generative AI spread after ChatGPT launched in late 2022, so earlier
figures aren't comparable.

In [ ]:
ACTIVITY = "Total - all activities"   # industry used by charts 5-7 (all industries combined)
SIZE = "10 or more"                   # firm size used by charts 5, 6 and 8 (the OECD survey's total)
MIN_YEAR = 2023                       # earliest year kept
ORANGE = "#eb6834"

ai = pd.read_csv(DATA_DIR / "oecd_ict_business_ai.csv")
ai = ai[ai["MEASURE"] == "G14_B"].rename(columns={
    "REF_AREA": "iso3", "Reference area": "country", "Economic activity": "activity",
    "Employment size class": "size", "TIME_PERIOD": "year"})

# Drop rows with no year (blank or footnote rows at the end of the export), then make year an integer
ai = ai.dropna(subset=["year"])
ai["year"] = ai["year"].astype(int)

# Post-ChatGPT only. This drops countries whose latest data predates 2023 (e.g. US, UK, Australia, Israel).
ai = ai[ai["year"] >= MIN_YEAR]

ai["share"] = ai["OBS_VALUE"] / 100   # percent -> fraction, so axes can use the ".0%" format
ai = ai[["iso3", "country", "activity", "size", "year", "share"]]

# Show the available labels (ACTIVITY and SIZE must match these exactly)
print("Industries:", sorted(ai["activity"].unique()))
print("Firm sizes:", sorted(ai["size"].unique()))
print("Years:", sorted(ai["year"].unique()))

# One row per country: its latest year with data, for the chosen industry and firm size
sel = ai[(ai["activity"] == ACTIVITY) & (ai["size"] == SIZE)].dropna(subset=["share"])
assert len(sel), "No rows for this ACTIVITY/SIZE - check the printed labels above"

latest = sel[sel["year"] == sel.groupby("iso3")["year"].transform("max")].copy()
# Safety check: if a country appears twice, some other dimension in the file needs filtering
assert not latest.duplicated("iso3").any(), latest[latest.duplicated("iso3", keep=False)]

# Countries whose latest year is older than the newest get the year in their label, e.g. "Korea (2024)"
newest = latest["year"].max()
latest["label"] = latest["country"].where(latest["year"] == newest,
                                          latest["country"] + " (" + latest["year"].astype(str) + ")")

# The EU-27 and OECD rows are averages, not countries: relabel them and mark them as a separate group.
# The EU code may be "EU27" or "EU27_2020" depending on the export, so match the start of the code.
is_eu = latest["iso3"].str.startswith("EU27")
is_oecd = latest["iso3"].str.startswith("OECD")
latest.loc[is_eu, "label"] = "EU-27 average"
latest.loc[is_oecd, "label"] = "OECD average"
latest["group"] = np.where(is_eu | is_oecd, "Average", "Country")

### Chart 5: Share of businesses using AI
The 10 highest and 10 lowest countries, with the EU-27 and OECD averages in gray for reference.
Value labels only on the extremes and averages; the axis covers the rest.

In [ ]:
countries_only = latest[latest["group"] == "Country"]

# The 10 highest + 10 lowest countries, plus the two averages
shown = pd.concat([countries_only.nlargest(10, "share"), countries_only.nsmallest(10, "share"),
                   latest[latest["group"] == "Average"]]).drop_duplicates("label")
ranked = shown.sort_values("share", ascending=False)["label"].tolist()   # bar order, highest first

y = alt.Y("label:N", sort=ranked, title=None, axis=alt.Axis(ticks=False, domain=False))
x = alt.X("share:Q", title="Share of businesses using AI", axis=alt.Axis(format=".0%"))

# Which bars get a value label: the 5 highest, the 5 lowest, and the averages
lab = shown[shown["label"].isin(countries_only.nlargest(5, "share")["label"])
            | shown["label"].isin(countries_only.nsmallest(5, "share")["label"])
            | (shown["group"] == "Average")]

bar_chart = (
    alt.Chart(shown).mark_bar(cornerRadiusEnd=4, height=14).encode(
        y=y, x=x,
        color=alt.Color("group:N", scale=alt.Scale(domain=["Country", "Average"], range=[BLUE, GRAY]),
                        legend=None))   # no legend: the subtitle explains the gray bars
    + alt.Chart(lab).mark_text(align="left", dx=4, fontSize=11, color="#52514e").encode(
        y=y, x=x, text=alt.Text("share:Q", format=".0%"))
).properties(width=480, height=alt.Step(20),
             title=alt.Title("Share of businesses using AI, all industries",
                             subtitle=[f"The 10 highest and 10 lowest of {len(countries_only)} countries. Firms with {SIZE} employees, "
                                       f"{newest} unless noted.", "Gray = EU-27 and OECD averages."]))
bar_chart.save("static-viz/figures/business_ai.png", scale_factor=2)
bar_chart

### Chart 6: GDP per capita vs. AI adoption
Each dot is a country. GDP per capita is adjusted for purchasing power parity (PPP), i.e. for price
differences between countries. It measures economic output per person, not household income.
Filled dots use the newest year; hollow dots use an older year (2023–2024).

In [ ]:
# Latest GDP per capita (PPP) for each country
wdi = pd.read_csv(DATA_DIR / "worldbank_wdi.csv")
gdp = (wdi[(wdi["indicator"] == "gdp_per_capita_ppp") & wdi["value"].notna()]
       .sort_values("year").groupby("iso3").last()["value"].rename("gdp"))

# Countries only (no EU/OECD averages), joined to GDP; countries without GDP data are dropped
scatter_df = (latest[latest["group"] == "Country"]
              .join(gdp, on="iso3").dropna(subset=["gdp"]))

# Category for the filled / hollow dots and their legend
older = f"{MIN_YEAR}–{newest - 1} data"
scatter_df["data_year"] = np.where(scatter_df["year"] == newest, f"{newest} data", older)

# Correlation between GDP per capita and AI adoption, printed and shown in the subtitle.
# Rough guide: about 0.5+ = moderate-to-strong, about 0.3 or less = weak.
r = np.corrcoef(scatter_df["gdp"], scatter_df["share"])[0, 1]
print(f"r = {r:.2f}")

# Only label notable countries; the crowded middle is left unlabelled to avoid overlaps.
# If two labels collide, remove one of the names here.
LABEL = ["Denmark", "Finland", "Korea", "Germany", "Norway", "Ireland", "Brazil",
         "Switzerland", "Romania", "Estonia", "Canada"]
scatter_df["show"] = scatter_df["country"].isin(LABEL)

x = alt.X("gdp:Q", title="GDP per capita, adjusted for purchasing power parity (current international $)",
          scale=alt.Scale(domain=[20_000, 150_000], nice=False),
          axis=alt.Axis(format="$,.0f", values=[25_000, 50_000, 75_000, 100_000, 125_000, 150_000]))
y = alt.Y("share:Q", title="Share of businesses using AI",
          scale=alt.Scale(domain=[0, 0.45]), axis=alt.Axis(format=".0%"))
base = alt.Chart(scatter_df).encode(x=x, y=y)

# Dots: blue outline always; fill is blue for the newest year and white (hollow) for older years
dots = base.mark_point(size=70, filled=True, stroke=BLUE, strokeWidth=1.5, opacity=1).encode(
    fill=alt.Fill("data_year:N", title=None,
                  scale=alt.Scale(domain=[f"{newest} data", older], range=[BLUE, "white"]),
                  legend=alt.Legend(orient="top", symbolStrokeColor=BLUE, symbolStrokeWidth=1.5,
                                    symbolType="circle")))

# Labels to the right of their dot
labels = base.transform_filter("datum.show").mark_text(
    align="left", dx=7, fontSize=10, color="#52514e").encode(text="label:N")
# Luxembourg sits at the right edge, so its label goes on the left of the dot
lux = base.transform_filter("datum.country == 'Luxembourg'").mark_text(
    align="right", dx=-7, fontSize=10, color="#52514e").encode(text="label:N")

scatter = (dots + labels + lux).properties(
    width=560, height=380,
    title=alt.Title(
        "Businesses in countries with higher GDP per capita tend to use more AI, but it explains only part of the gap",
        subtitle=[f"Firms with {SIZE} employees, all industries, latest year ({MIN_YEAR} or later). "
                  f"Correlation with GDP per capita: r = {r:.2f}.",
                  "GDP per capita is economic output per person (adjusted for price differences between countries), not household income.",
                  "Luxembourg's and Ireland's GDP is inflated by cross-border workers and multinational profits."]))
scatter.save("static-viz/figures/gdp_ai_scatter.png", scale_factor=2)
scatter

### Chart 7: AI use in small vs. large firms
Small = 10–49 employees, large = 250 or more (mid-sized firms are left out for clarity).
Shows the 10 countries with the highest and the 10 with the lowest large-firm adoption.

In [ ]:
small, large = "From 10 to 49", "250 or more"   # labels as they appear in the OECD file
S, L = "Small firms", "Large firms"             # labels shown on the chart
SMALL_C, LARGE_C = "#5fc3b7", "#0b6e66"         # light / dark teal: lighter = smaller firms

# Countries only (no EU/OECD averages), both size classes
by_size = ai[(ai["activity"] == ACTIVITY) & ai["size"].isin([small, large])
             & ~ai["iso3"].str.startswith(("EU27", "OECD"))].dropna(subset=["share"])

# One row per country-year with small and large columns; keep years where both exist,
# then each country's latest such year (so a country isn't dropped if one size class is missing)
by_size = (by_size.pivot_table(index=["country", "year"], columns="size", values="share")
           .dropna().reset_index())
by_size = by_size[by_size["year"] == by_size.groupby("country")["year"].transform("max")]
by_size = by_size.rename(columns={small: S, large: L})

# Year in the label for countries with older data. Uses its own variable name so it can't
# overwrite `newest`, which charts 5 and 6 rely on.
size_newest = by_size["year"].max()
by_size["label"] = by_size["country"].where(by_size["year"] == size_newest,
                                            by_size["country"] + " (" + by_size["year"].astype(str) + ")")

# Keep the 10 highest and 10 lowest by large-firm adoption
n_size = len(by_size)
by_size = pd.concat([by_size.nlargest(10, L), by_size.nsmallest(10, L)]).drop_duplicates("label")
order = by_size.sort_values(L, ascending=False)["label"].tolist()

y = alt.Y("label:N", sort=order, title=None, axis=alt.Axis(ticks=False, domain=False))
# Axis runs to at least 85%, further if the top dot needs the room
x_scale = alt.Scale(domain=[0, max(0.85, by_size[L].max() + 0.03)], nice=False)

# Gray line between the two values, then the two dots (same structure as chart 3)
rule = alt.Chart(by_size).mark_rule(color="#c9c8c2", strokeWidth=2).encode(
    y=y, x=alt.X(f"{S}:Q", scale=x_scale), x2=f"{L}:Q")
dots = alt.Chart(by_size).transform_fold([S, L], as_=["size", "share"]).mark_circle(size=90, opacity=1).encode(
    y=y,
    x=alt.X("share:Q", scale=x_scale, title="Share of businesses using AI",
            axis=alt.Axis(format=".0%", values=[0, 0.2, 0.4, 0.6, 0.8])),
    color=alt.Color("size:N", title=None, legend=alt.Legend(orient="top"),
                    scale=alt.Scale(domain=[S, L], range=[SMALL_C, LARGE_C])))

size_chart = (rule + dots).properties(
    width=480, height=alt.Step(18),
    title=alt.Title(
        "Large firms are far more likely to use AI than small ones, in every country",
        subtitle=[f"Share of businesses using AI, all industries, {size_newest} unless noted. "
                  f"The 10 highest and 10 lowest of {n_size} countries, sorted by large-firm adoption.",
                  "Small = 10–49 employees, large = 250 or more."]))
size_chart.save("static-viz/figures/ai_firm_size.png", scale_factor=2)
size_chart

### Chart 8: AI use by industry
Average share of businesses using AI in each industry, across the countries in the OECD survey
(each country counts equally). "All industries" is shown in gray as a reference.
Note this is an average of the survey countries, mostly European, not a worldwide figure.

In [ ]:
# Short industry names for the axis
SHORT = {
    "Total - all activities": "All industries",
    "Information and communication": "Info & comms",
    "Professional, scientific and technical activities": "Professional & scientific",
    "Financial and insurance activities": "Finance & insurance",
    "Administrative and support service activities": "Admin & support",
    "Wholesale trade, except of motor vehicles and motorcycles": "Wholesale",
    "Retail trade, except of motor vehicles and motorcycles": "Retail",
    "Real estate activities": "Real estate",
    "Manufacturing": "Manufacturing",
    "Construction": "Construction",
    "Transportation and storage": "Transport",
    "Accommodation and food service activities": "Hotels & food",
}

# Countries only, firms with 10+ employees
ind = ai[(ai["size"] == SIZE) & ~ai["iso3"].str.startswith(("EU27", "OECD"))].dropna(subset=["share"]).copy()

# One year per country: its latest year with an all-industries figure (so each country's industries match)
yr = ind[ind["activity"] == "Total - all activities"].groupby("iso3")["year"].max()
ind = ind[ind["year"] == ind["iso3"].map(yr)].copy()
ind["industry"] = ind["activity"].map(SHORT)
n_countries = ind["iso3"].nunique()

# Drop industries reported by fewer than half the countries: their average would be unrepresentative.
# Mention anything printed here in the write-up.
coverage = ind.groupby("industry")["iso3"].nunique()
dropped = coverage[coverage < n_countries / 2].index.tolist()
print("Dropped for low coverage:", dropped)
ind = ind[~ind["industry"].isin(dropped)]

# Average across countries for each industry
avg = ind.groupby("industry", as_index=False)["share"].mean()
avg["is_total"] = avg["industry"] == "All industries"
ind_order = avg.sort_values("share", ascending=False)["industry"].tolist()   # highest first

y = alt.Y("industry:N", sort=ind_order, title=None, axis=alt.Axis(ticks=False, domain=False))
x = alt.X("share:Q", title="Average share of businesses using AI", axis=alt.Axis(format=".0%"))

industry_chart = (
    alt.Chart(avg).mark_bar(cornerRadiusEnd=4, height=16).encode(
        y=y, x=x,
        color=alt.condition("datum.is_total", alt.value(GRAY), alt.value(BLUE)))   # gray = all industries
    + alt.Chart(avg).mark_text(align="left", dx=4, fontSize=11, color="#52514e").encode(
        y=y, x=x, text=alt.Text("share:Q", format=".0%"))
).properties(
    width=480, height=alt.Step(24),
    title=alt.Title(
        "Information and communication firms are by far the heaviest AI users",
        subtitle=[f"Average share of businesses with {SIZE} employees using AI, across {n_countries} countries in the OECD survey, "
                  f"{MIN_YEAR} or later.",
                  "Each country counts equally. Gray = all industries combined."]))
industry_chart.save("static-viz/figures/ai_by_industry.png", scale_factor=2)
industry_chart

## Export SVGs for the PDF report
`make_report.py` embeds these SVG files in `report.pdf`. SVGs are vector graphics, so they stay sharp
at any zoom. Re-run this cell whenever a chart changes, then re-run `make_report.py`.

In [ ]:
charts = {"clerical_map": map_chart, "occupation_heatmap": heatmap, "sex_clerical_jobs": dumbbell,
          "clerical_lines": line_chart, "business_ai": bar_chart, "gdp_ai_scatter": scatter,
          "ai_firm_size": size_chart, "ai_by_industry": industry_chart}
for name, chart in charts.items():
    chart.save(f"static-viz/figures/{name}.svg")
print("Saved", len(charts), "SVGs to figures/")